# Colab Runner: PPO + Imitation Learning (+ extended studies & SAC)

**Group members:** Marco De Palma, Em Echeverria, Leah Sarouphin, Juan Jose Rincon Briceño, Matteo Mainetti

Runs the whole project end to end on Colab, with every artifact persisted to Google Drive so disconnects do not lose work. Each section says **what** it does and **why** the final hyperparameters were chosen.

**Runtime:** use a **GPU** runtime (Runtime -> Change runtime type -> GPU). It speeds up the from-scratch BC and SAC; PPO is CPU-bound (the MuJoCo sim is not GPU-accelerated) so a GPU neither helps nor hurts it. Long runs need Colab Pro or an active tab to avoid idle disconnects.

**Order:** run section 1 (setup) first; after the numpy restart it asks for, re-run 1.1 and 1.3, then go top to bottom. Stages write to Drive and resume from checkpoints, so you can stop and continue any time.

## 1. Setup: clone, install, persist

The pipeline uses the `imitation` library, which pins gymnasium 0.29 and therefore numpy < 2. Colab ships numpy 2.x, so installing `requirements.txt` downgrades numpy and needs a **one-time runtime restart**. The dependency-resolver warnings about jax / transformers / opencv are harmless (those packages are unused here).

### 1.1 Clone the code

In [ ]:
REPO = 'https://github.com/em-ech/rl-ppo-imitation-learning.git'
BRANCH = 'main'
CODE_DIR = '/content/GroupProject'
import os, sys, shutil
os.chdir('/content')  # never stand inside the dir we delete
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)
!git clone -q --branch {BRANCH} {REPO} {CODE_DIR}
sys.path.insert(0, CODE_DIR); os.chdir(CODE_DIR)
print('code in', CODE_DIR, '->', sorted(os.listdir(CODE_DIR))[:8])

### 1.2 Install pinned dependencies, then restart once

Run this cell, then **Runtime -> Restart session**, then re-run 1.1 and 1.3 and continue from section 2. Skip this cell on the second pass (the install persists across a restart).

In [ ]:
!pip -q install -r /content/GroupProject/requirements.txt
print('Install done. NOW: Runtime -> Restart session, then re-run 1.1 and 1.3.')

### 1.3 Mount Drive, configure persistence, sanity check

`PROJECT_DATA_ROOT` -> Drive sends `models/ data/ outputs/ logs/` to Drive, so checkpoints (every 100k steps) and figures survive disconnects. The `!python` calls below inherit this env var.

In [ ]:
import sys, os  # re-assert after a possible restart
sys.path.insert(0, '/content/GroupProject'); os.chdir('/content/GroupProject')
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/rl_project'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ['PROJECT_DATA_ROOT'] = DRIVE_ROOT
from src import config
print('device:', config.device(), '| models ->', config.MODELS_DIR)
import gymnasium as gym
for e in ['Walker2d-v4', 'Ant-v4', 'HalfCheetah-v4']:
    gym.make(e).reset(seed=0); print('env ok:', e)

## 2. Stage 1 - PPO experts (M1)

**What:** train one PPO expert per environment; it is the oracle every student imitates. Targets: Walker2d > 3000, Ant > 4000.

**Why these settings (the key finding):** tuned recipes do **not** transfer across environments, but `VecNormalize` was the decisive lever on both.
- **Walker2d:** the standard config (`n_steps=2048`, `batch=64`, `lr=3e-4` linear) + VecNormalize, 8M steps -> ~6043. The rl-zoo3 `tuned_walker` profile actually *failed* here (1640).
- **Ant:** the standard config plateaued (~2850), so we used the rl-zoo3 Optuna profile (`tuned_ant`: `n_steps=512`, `batch=32`, `lr=1.9e-5`, `gamma=0.98`) + VecNormalize, 10M steps -> ~6293.

Slow (Walker ~30-40 min, Ant ~90 min+). Append `resume` to continue from the last 100k checkpoint after a disconnect.

In [ ]:
!cd /content/GroupProject && python train_expert.py Walker2d-v4 8000000 4 norm default

In [ ]:
!cd /content/GroupProject && python train_expert.py Ant-v4 10000000 1 norm tuned_ant
# resume after a disconnect:
# !cd /content/GroupProject && python train_expert.py Ant-v4 10000000 1 norm tuned_ant resume

## 3. Stage 2 - demonstration dataset + EDA (M2)

**What:** roll out the deterministic expert and record (observation, action) pairs, then run the EDA and a quality gate.

**Why these settings:** we record the **deterministic** policy mean (not a stochastic sample) to give the student clean labels, and collect **100** episodes (the spec floor is 50). The quality gate requires >= 90% of episodes above two-thirds of the mean return; an inconsistent expert cannot be cloned cleanly (an earlier 2850-return Ant expert failed this gate, which is why we retrained it).

In [ ]:
!cd /content/GroupProject && python collect_demos.py Walker2d-v4 100
!cd /content/GroupProject && python collect_demos.py Ant-v4 100

## 4. Stage 3 - Behavioural Cloning (M3, M4, M5, M8)

**What:** train the BC student two ways (the `imitation` library and a from-scratch PyTorch MLP), plus an epoch sweep, a dataset-size ablation (5 seeds), and an architecture sweep.

**Why these settings:**
- The experts use VecNormalize, so BC **trains and evaluates on the expert's normalised observations**; learning from raw observations fails (an early bug gave Walker2d BC only ~1246).
- Scratch BC uses validation-loss **early stopping** (150-epoch ceiling); library BC uses 100 epochs. A fixed 50-epoch budget under-trained the strong Walker2d student (~47% of expert); more budget lifted it to ~95%.

Use a GPU runtime here. ~40 min per environment.

In [ ]:
!cd /content/GroupProject && python bc_experiments.py Walker2d-v4
!cd /content/GroupProject && python arch_sweep.py     Walker2d-v4
!cd /content/GroupProject && python bc_experiments.py Ant-v4
!cd /content/GroupProject && python arch_sweep.py     Ant-v4

## 5. Stage 4 - DAgger (M7)

**What:** roll out the current student, query the expert on the visited states, aggregate, and retrain, fighting the covariate shift that plain BC suffers.

**Why these settings:** the comparison must hold **training effort fixed**. An early run used only 4 BC epochs per iteration and looked far worse than BC; the fair configuration is **12 iterations x 25 BC epochs**, which matches BC's ~100-epoch budget and reaches expert level. ~6-10 min each.

In [ ]:
!cd /content/GroupProject && python dagger_run.py Walker2d-v4 Walker2d-v4
!cd /content/GroupProject && python dagger_run.py Ant-v4      Ant-v4

## 6. Stage 5 - imitation as PPO pretraining (the central question)

**What:** compare PPO from scratch against PPO warm-started from the BC and DAgger policies, as eval return vs environment timesteps at a fixed 1.5M budget. This answers whether imitation reduces PPO's sample complexity.

**Why these settings:** the 1.5M-step budget is far below the expert's 8-10M, so it exposes sample efficiency. We pass each env its own profile (`default` for Walker2d, `tuned_ant` for Ant) and the matching n_envs, so the warm-started and from-scratch runs are directly comparable. ~20-40 min per env.

In [ ]:
!cd /content/GroupProject && python pretraining.py Walker2d-v4 Walker2d-v4 default   1500000 4
!cd /content/GroupProject && python pretraining.py Ant-v4      Ant-v4      tuned_ant 1500000 1

## 7. Extended requirements - E1 noisy expert, E2 obs normalisation (bonus)

**What:** two from-scratch BC ablations, 5 seeds each, both environments.
- **E1:** add zero-mean Gaussian noise to the recorded actions at increasing std (clipped to the [-1,1] torque range) and find where the student collapses.
- **E2:** train BC with vs without standardising observations to zero mean / unit variance, comparing convergence and final return.

**Why these settings:** 5 seeds matches the dataset-size and architecture sweeps so the curves carry error bars; the runs are **resumable** (checkpointed per noise level / condition) so a disconnect never recomputes finished work. ~15-30 min per env. Figures land in notebook 06.

**Finding:** Walker2d is fragile (collapses below half-expert by sigma=0.05; raw-obs BC fails, a 4x gap) while Ant is robust to both, reinforcing RQ6.

In [ ]:
!cd /content/GroupProject && python noise_sweep.py   Walker2d-v4
!cd /content/GroupProject && python norm_ablation.py Walker2d-v4
!cd /content/GroupProject && python noise_sweep.py   Ant-v4
!cd /content/GroupProject && python norm_ablation.py Ant-v4

## 8. Bonus - SAC off-policy experts (vs on-policy PPO)

**What:** train SAC experts and compare against the PPO experts, highlighting off-policy sample efficiency. Ant is the in-brief comparison; HalfCheetah is off-brief and the clean path past an 8000 score.

**Why these settings:** SAC is off-policy and update-bound, so unlike PPO it benefits from a **GPU** (1-2h per env). It uses **no VecNormalize** (running stats would corrupt the replay buffer), so it also produces raw-observation demonstrations. The hyperparameters are the rl-zoo3 MuJoCo recipe (`lr=7.3e-4`, `gamma=0.98`, `tau=0.02`, `train_freq=8`, `gradient_steps=8`, gSDE), net `[400,300]` for Ant and `[256,256]` for HalfCheetah, 3M steps. Realistic returns: Ant ~5000-7000 (PPO got 6293), HalfCheetah ~9000-11000. Append `resume` to continue from the last checkpoint; `best_model` is saved by eval return, so a restart never loses your best policy (back up `best_model.zip` first if you want to be safe).

In [ ]:
!cd /content/GroupProject && python train_sac.py Ant-v4 3000000 tuned_ant
# resume after a disconnect (continues from the last Drive checkpoint):
# !cd /content/GroupProject && python train_sac.py Ant-v4 3000000 tuned_ant resume

In [ ]:
!cd /content/GroupProject && python train_sac.py HalfCheetah-v4 3000000 tuned_halfcheetah
# resume:
# !cd /content/GroupProject && python train_sac.py HalfCheetah-v4 3000000 tuned_halfcheetah resume

## Recover after a disconnect or session restart

Code is on GitHub; trained artifacts are on Drive under `MyDrive/rl_project/` (`models/`, `outputs/`, `logs/`, every 100k-step checkpoint). After a restart, re-run 1.1 and 1.3 (re-clone + re-mount, no need to re-install or restart again), then re-run the stage you were on, appending `resume` where the cell shows it. A restart only loses progress that was not on Drive, so always run cell 1.3 (which sets `PROJECT_DATA_ROOT`) before training.

The five core notebooks (01-05) and the extended notebook (06) display all figures and the full discussion; this runner just produces the artifacts.